In [ ]:
import torch
from ultralytics import YOLO

yolo = YOLO("yolov5nu.pt")
net = yolo.model

In [ ]:
print(net)

In [ ]:
def flatten_atomic(module, prefix=""):
    atomic_layers = []
    for name, sub in module.named_children():
        full_name = f"{prefix}.{name}" if prefix else name
        if list(sub.children()):  # has submodules
            atomic_layers.extend(flatten_atomic(sub, prefix=full_name))
        else:  # atomic layer
            atomic_layers.append({"full_name": full_name, "model": sub})
    return atomic_layers

atomic_layers = flatten_atomic(net.model)
atomic_layers

In [ ]:
state_dict = net.state_dict()
weights_list = [{ "name": k, "weight": v.cpu()} for k, v in state_dict.items()]
# Flattened weights with shapes
weights_list_shapes = [
    {"name": k, "shape": tuple(v.shape)}
    for k, v in net.state_dict().items()
]

# Print nicely
for w in weights_list_shapes:
    print(w)

In [ ]:
# Prepare a copy of atomic_layers with weights
aligned_layers = []

for layer in atomic_layers:
    full_name = layer['full_name']
    matched_weights = []

    # Rule: name contains "model." + full_name + "."
    prefix = f"model.{full_name}."
    for w in weights_list:
        if w['name'].startswith(prefix):
            matched_weights.append(w)  # keep as is

    # Save aligned entry
    layer_entry = layer.copy()
    layer_entry['weights'] = matched_weights  # can be empty list if no match
    aligned_layers.append(layer_entry)


In [ ]:
def vis_layer(layer):
    print(f"Layer: {layer['full_name']}, name: {type(layer['model']).__name__}, ly: {layer['model']}")
    if layer['weights']:
        for w in layer['weights']:
            print(f"  weight name: {w['name']}, shape: {tuple(w['weight'].shape)}")
    else:
        print("  no weights")
    print("--------------------")

# Visualize first 10 layers
for i in range(10):
    vis_layer(aligned_layers[i])

In [ ]:
len(aligned_layers)

In [ ]:
atomic_graph = {layer['full_name']: [] for layer in atomic_layers}
atomic_graph['0.conv'] = [-1]
atomic_graph

In [ ]:
layer_graph = {}

for i, m in enumerate(net.model):
    if hasattr(m, 'f'):
        # ensure f is a list
        if isinstance(m.f, int):
            from_layers = [m.f]
        else:
            from_layers = m.f if len(m.f) > 0 else [-1]
    else:
        from_layers = [-1]  # default for layers without f
    layer_graph[i] = from_layers

layer_graph

In [ ]:
for i in list(layer_graph.keys()):
    for idx, j in enumerate(layer_graph[i]):
        if j == -1 and layer_graph[i] != 0:
            layer_graph[i][idx] = i+j
layer_graph

In [ ]:
atomic_port = {}

for atomic in atomic_layers:
    full_name = atomic['full_name']
    layer_num = int(full_name.split('.')[0])  # get the first number before the first dot

    if layer_num not in atomic_port:
        # first atomic layer for this layer -> entry
        atomic_port[layer_num] = [full_name, full_name]  # [entry, exit]
    else:
        # update exit as we encounter more atomic layers in this layer
        atomic_port[layer_num][1] = full_name

# convert the list to tuple for (entry, exit)
atomic_port = {k: tuple(v) for k, v in atomic_port.items()}

# print the result
for k in sorted(atomic_port):
    print(k, atomic_port[k])

In [ ]:
for layer, from_layers in layer_graph.items():
    for from_layer in from_layers:
        if from_layer != -1:
            port = atomic_port[layer][0]
            exit_port = atomic_port[from_layer][1]  # 1 = exit
            atomic_graph[port].append(exit_port)
atomic_graph

In [ ]:
keys_in_order = list(atomic_graph.keys())

prev_key = None
for key in keys_in_order:
    # Check if this key is the entry in atomic_port
    is_entry = any(key == entry for entry, _ in atomic_port.values())
    
    if not is_entry and prev_key is not None:
        atomic_graph[key].append(prev_key)
    
    prev_key = key
atomic_graph

In [ ]:
from graphviz import Digraph


dot = Digraph(comment='Atomic Graph')

# Add nodes
for node in atomic_graph.keys():
    dot.node(node, node)

# Add edges
for node, inputs in atomic_graph.items():
    for inp in inputs:
        if inp != -1:  # skip -1 input placeholder
            dot.edge(inp, node)
dot.render(filename='atomic_graph', cleanup=True)

In [ ]:
atomic_layers_names = [layer['full_name'] for layer in atomic_layers]
atomic_graph
atomic_layers_names

In [ ]:
# Suppose atomic_graph is your dictionary with full names as keys and dependencies
# Example: {'0.conv': [-1], '0.bn': ['0.conv'], ... }

# Map full names to integer indices
name_to_idx = {name: idx for idx, name in enumerate(atomic_layers_names)}

# Convert graph to integer indices
graph = {}

for node, parents in atomic_graph.items():
    # Convert node name to its index
    node_idx = name_to_idx[node]
    
    # Convert parents to indices; handle -1 as a special input node
    parent_indices = []
    for p in parents:
        if isinstance(p, int) and p == -1:
            parent_indices.append(-1)  # keep input node as -1
        else:
            parent_indices.append(name_to_idx[p])
    
    graph[node_idx] = parent_indices

graph

In [ ]:
from graphviz import Digraph

dot = Digraph(comment='Integer Graph')

# Add nodes
for node_idx in graph.keys():
    dot.node(str(node_idx), str(node_idx))  # Graphviz nodes must be strings

# Add edges
for node_idx, parent_indices in graph.items():
    for parent_idx in parent_indices:
        if parent_idx != -1:  # skip input placeholder
            dot.edge(str(parent_idx), str(node_idx))
dot.render(filename='graph', cleanup=True)

In [ ]:
aligned_layers_mapped = []
atomic_name_to_idx = {name: idx for idx, name in enumerate(atomic_layers_names)}
for layer in aligned_layers:
    # Map full_name → node integer
    full_name = layer['full_name']
    node_idx = atomic_name_to_idx.get(full_name, -1)
    
    new_layer = layer.copy()
    new_layer['node'] = node_idx
    del new_layer['full_name']
    
    # Keep only last part after last dot
    new_weights = []
    for w in layer['weights']:
        short_name = w['name'].split('.')[-1]  # take last segment
        new_w = w.copy()
        new_w['name'] = short_name
        new_weights.append(new_w)
    
    new_layer['weights'] = new_weights
    aligned_layers_mapped.append(new_layer)

In [ ]:
for layer in aligned_layers_mapped:
    print(f"Node: {layer['node']}, Model: {type(layer['model']).__name__}")
    for w in layer['weights']:
        print(f"  Weight name: {w['name']}, shape: {tuple(w['weight'].shape)}")
    print('-'*50)

In [ ]:
operations = [type(layer['model']).__name__ for layer in aligned_layers_mapped]
operations 

In [ ]:
config_dict = {}

for idx, layer in enumerate(aligned_layers_mapped):
    op = layer['model']
    op_type = type(op).__name__
    
    if op_type == "Conv2d":
        # Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias)
        in_ch = op.in_channels
        out_ch = op.out_channels
        k_h, k_w = op.kernel_size if isinstance(op.kernel_size, tuple) else (op.kernel_size, op.kernel_size)
        s_h, s_w = op.stride if isinstance(op.stride, tuple) else (op.stride, op.stride)
        p_h, p_w = op.padding if isinstance(op.padding, tuple) else (op.padding, op.padding)
        bias = op.bias is not None
        config_dict[idx] = [in_ch, out_ch, k_h, k_w, s_h, s_w, p_h, p_w, bias]
        
    elif op_type == "BatchNorm2d":
        # BatchNorm2d(num_features, eps, momentum, affine, track_running_stats)
        num_feat = op.num_features
        eps = op.eps
        momentum = op.momentum
        affine = op.affine
        track_stats = op.track_running_stats
        config_dict[idx] = [num_feat, eps, momentum, affine, track_stats]
        
    elif op_type == "SiLU":
        # SiLU(inplace)
        inplace = op.inplace
        config_dict[idx] = [inplace]
        
    elif op_type == "MaxPool2d":
        # MaxPool2d(kernel_size, stride, padding, dilation, ceil_mode)
        k_h, k_w = op.kernel_size if isinstance(op.kernel_size, tuple) else (op.kernel_size, op.kernel_size)
        s_h, s_w = op.stride if isinstance(op.stride, tuple) else (op.stride, op.stride)
        p_h, p_w = op.padding if isinstance(op.padding, tuple) else (op.padding, op.padding)
        d_h, d_w = op.dilation if isinstance(op.dilation, tuple) else (op.dilation, op.dilation)
        config_dict[idx] = [k_h, k_w, s_h, s_w, p_h, p_w, d_h, d_w, op.ceil_mode]
        
    elif op_type == "Upsample":
        # Upsample(scale_factor, mode)
        scale_h, scale_w = op.scale_factor if isinstance(op.scale_factor, tuple) else (op.scale_factor, op.scale_factor)
        config_dict[idx] = [scale_h, scale_w, op.mode]
        
    elif op_type == "Concat":
        # Concat usually no parameters
        config_dict[idx] = []
        
    else:
        # Unknown op fallback
        config_dict[idx] = []

config_dict

In [ ]:
config = [config_dict[idx] for idx in range(len(aligned_layers_mapped))]
config

In [ ]:
weights = [
    [w['weight'] for w in layer['weights']]  # list of tensors for this node
    for layer in aligned_layers_mapped
]
weights

In [ ]:
for idx, node_weights in enumerate(weights):
    print(f"Node {idx}:")
    if node_weights:  # if node has weights
        for w_idx, w in enumerate(node_weights):
            print(f"  Weight {w_idx} shape: {tuple(w.shape)}")
    else:
        print("  No weights")
    print("-" * 50)

In [ ]:
# Conv2d: 
# Config = [in_ch, out_ch, k_h, k_w, stride_h, stride_w, pad_h, pad_w, bias_flag]; 
# Weight = [weight(out_ch x in_ch x kH x kW), bias(out_ch) if exists];

# BatchNorm2d: Config = [num_features, eps, momentum, affine_flag, track_running_stats_flag];
# Weight = [weight(num_features), bias(num_features), running_mean(num_features), running_var(num_features)]

# SiLU: Config = [inplace_flag];
# Weight = []

# MaxPool2d: Config = [kernel_h, kernel_w, stride_h, stride_w, pad_h, pad_w, dilation_h, dilation_w, ceil_mode_flag];
# Weight = []

# Upsample: Config = [scale_h, scale_w, mode];
# Weight = []

# Concat: Config = [];
# Weight = []

graph
operations
config
weights


np.save("graph.npy", graph)
np.save("operations.npy", operations)
np.save("config.npy", config)
np.save("weights.npy", weights)